# ETL Pipeline — complex_report

This notebook ingests data from the original Tableau data sources
into Fabric Lakehouse Delta tables.

**Generated:** 2026-03-05 10:46
**Source:** TableauToFabric migration tool


In [ ]:
# ── Setup ──────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime

# Fabric Lakehouse is automatically mounted
LAKEHOUSE_NAME = "complex_report_Lakehouse"
print(f"ETL Pipeline started: {datetime.now()}")


## Data Ingestion

Loading 5 table(s) from source systems.


### Table: `Orders` (SQL Server)

In [ ]:
# Read from SQL Server
jdbc_url = "jdbc:sqlserver://sql-finance.contoso.com;databaseName=FinanceDB"
df_orders = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "Orders") \
    .option("user", "<username>") \
    .option("password", "<password>") \
    .load()

# Preview
df_orders.printSchema()
print(f"Rows: {df_orders.count()}")

# Write to Lakehouse Delta table
df_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("orders")
print(f"✓ Written to Lakehouse: orders")


### Table: `Customers` (SQL Server)

In [ ]:
# Read from SQL Server
jdbc_url = "jdbc:sqlserver://sql-finance.contoso.com;databaseName=FinanceDB"
df_customers = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "Customers") \
    .option("user", "<username>") \
    .option("password", "<password>") \
    .load()

# Preview
df_customers.printSchema()
print(f"Rows: {df_customers.count()}")

# Write to Lakehouse Delta table
df_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customers")
print(f"✓ Written to Lakehouse: customers")


### Table: `Products` (SQL Server)

In [ ]:
# Read from SQL Server
jdbc_url = "jdbc:sqlserver://sql-finance.contoso.com;databaseName=FinanceDB"
df_products = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "Products") \
    .option("user", "<username>") \
    .option("password", "<password>") \
    .load()

# Preview
df_products.printSchema()
print(f"Rows: {df_products.count()}")

# Write to Lakehouse Delta table
df_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("products")
print(f"✓ Written to Lakehouse: products")


### Table: `Regions` (SQL Server)

In [ ]:
# Read from SQL Server
jdbc_url = "jdbc:sqlserver://sql-finance.contoso.com;databaseName=FinanceDB"
df_regions = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "Regions") \
    .option("user", "<username>") \
    .option("password", "<password>") \
    .load()

# Preview
df_regions.printSchema()
print(f"Rows: {df_regions.count()}")

# Write to Lakehouse Delta table
df_regions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("regions")
print(f"✓ Written to Lakehouse: regions")


### Table: `Targets` (Excel)

In [ ]:
# Read Excel file (requires com.crealytics:spark-excel)
df_targets = spark.read \
    .format("com.crealytics.spark.excel") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("dataAddress", "\'Targets\'!A1") \
    .load("Files/\\\\fileserver\\data\\Budget_2024.xlsx")

# Preview
df_targets.printSchema()
print(f"Rows: {df_targets.count()}")

# Write to Lakehouse Delta table
df_targets.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("targets")
print(f"✓ Written to Lakehouse: targets")


## Custom SQL Queries

### MonthlySummary

In [ ]:
# Custom SQL: MonthlySummary
custom_query = """
SELECT
              YEAR(OrderDate)  AS OrderYear,
              MONTH(OrderDate) AS OrderMonth,
              Category,
              SUM(Revenue)     AS TotalRevenue,
              SUM(Cost)        AS TotalCost,
              COUNT(*)         AS OrderCount
          FROM Orders
          GROUP BY YEAR(OrderDate), MONTH(OrderDate), Category
"""

# TODO: Configure JDBC connection
# df_monthlysummary = spark.read.format("jdbc").option("query", custom_query).load()
# df_monthlysummary.write.format("delta").mode("overwrite").saveAsTable("monthlysummary")


## Summary

In [ ]:
# ── Summary ────────────────────────────────────────────
print("\n" + "=" * 60)
print("ETL Pipeline Complete")
print("=" * 60)
print(f"Tables loaded: 5")
print(f"Completed: {datetime.now()}")

# List all Delta tables in Lakehouse
print("\nLakehouse tables:")
for t in spark.catalog.listTables():
    print(f"  - {t.name} ({t.tableType})")
